# Downgrade quality: synthetic power-law + real d10 template

This notebook compares `ud_grade`, `harmonic_ud_grade`, and `smoothing + ud_grade`.

For the synthetic case, quality is measured against a direct reference map synthesized at the target pixelization from the same input spectrum realization.

In [ ]:
from pathlib import Path
import urllib.request

import numpy as np
import healpy as hp
import matplotlib.pyplot as plt


In [ ]:
def _round_sig(x, sig=2):
    if x == 0 or not np.isfinite(x):
        return float(x)
    return float(np.round(x, sig - int(np.floor(np.log10(abs(x)))) - 1))

def rounded_limits(*maps, q=(0.5, 99.5)):
    vals = np.concatenate([m[np.isfinite(m)] for m in maps])
    lo, hi = np.percentile(vals, q)
    vmin = _round_sig(float(lo), sig=2)
    vmax = _round_sig(float(hi), sig=2)
    if vmin == vmax:
        vmax = vmin + 1.0
    return vmin, vmax

def proj_panels(maps, titles, cmap='RdBu_r', q=(0.5, 99.5), ncol=3, xsize=2200, symmetric=False):
    vmin, vmax = rounded_limits(*maps, q=q)
    if symmetric:
        lim = max(abs(vmin), abs(vmax))
        vmin, vmax = -lim, lim
    print(f'vmin={vmin}, vmax={vmax}')
    n = len(maps)
    ncol = min(ncol, n)
    nrow = int(np.ceil(n / ncol))
    plt.figure(figsize=(6.8*ncol, 4.8*nrow))
    for i, (m, t) in enumerate(zip(maps, titles), start=1):
        hp.projview(
            m,
            sub=(nrow, ncol, i),
            title=t,
            min=vmin,
            max=vmax,
            cmap=cmap,
            graticule=True,
            xsize=xsize,
            cb_orientation='horizontal',
        )
    plt.tight_layout()

def compare_to_ref(m, m_ref, cl, cl_ref, ell_min=2):
    good = np.isfinite(m) & np.isfinite(m_ref)
    m0 = m[good]
    r0 = m_ref[good]
    rmse_rel = np.sqrt(np.mean((m0-r0)**2)) / np.std(r0)
    mae_rel = np.mean(np.abs(m0-r0)) / np.std(r0)
    corr = np.corrcoef(m0, r0)[0, 1]
    spec_rel_l2 = np.linalg.norm(cl[ell_min:] - cl_ref[ell_min:]) / np.linalg.norm(cl_ref[ell_min:])
    return {
        'rmse_rel_std': float(rmse_rel),
        'mae_rel_std': float(mae_rel),
        'map_corr': float(corr),
        'spec_rel_l2': float(spec_rel_l2),
    }

def moll_diff_panels(diff_maps, titles, unit='', q=(0.2, 99.8)):
    vmin, vmax = rounded_limits(*diff_maps, q=q)
    lim = max(abs(vmin), abs(vmax))
    vmin, vmax = -lim, lim
    print(f'moll diff scale: vmin={vmin}, vmax={vmax}')
    n = len(diff_maps)
    fig = plt.figure(figsize=(6.8*n, 5.0))
    for i, (m, t) in enumerate(zip(diff_maps, titles), start=1):
        hp.mollview(
            m,
            fig=fig.number,
            sub=(1, n, i),
            title=t,
            min=vmin,
            max=vmax,
            cmap='RdBu_r',
            unit=unit,
            cbar=True,
        )
    plt.tight_layout()


## 1) Synthetic power-law map with reference at target NSIDE

In [ ]:
nside_in = 256
nside_out = 32
lmax_in = 3 * nside_in - 1
lmax_out = 3 * nside_out - 1

ell = np.arange(lmax_in + 1)
cl = np.zeros(lmax_in + 1)
cl[2:] = (ell[2:] / 80.0) ** (-2.7)
cl /= cl[80]

np.random.seed(1234)
alm_in = hp.synalm(cl, lmax=lmax_in)
m_in = hp.alm2map(alm_in, nside=nside_in, lmax=lmax_in, pixwin=False)

# Reference map: directly synthesize at output NSIDE from the same realization
alm_ref = hp.resize_alm(alm_in, lmax_in, lmax_in, lmax_out, lmax_out)
m_ref = hp.alm2map(alm_ref, nside=nside_out, lmax=lmax_out, pixwin=False)

In [ ]:
m_ud = hp.ud_grade(m_in, nside_out=nside_out)
m_harm_default = hp.harmonic_ud_grade(m_in, nside_out=nside_out)
m_harm_lmax_out = hp.harmonic_ud_grade(m_in, nside_out=nside_out, lmax=lmax_out)
m_smooth_ud = hp.ud_grade(hp.smoothing(m_in, fwhm=np.radians(30/60)), nside_out=nside_out)

In [ ]:
proj_panels(
    [m_ref, m_ud, m_harm_default, m_harm_lmax_out, m_smooth_ud],
    ['reference', 'ud_grade', 'harmonic (default)', 'harmonic (lmax=3N-1)', 'smoothing + ud'],
    cmap='RdBu_r',
    q=(0.5, 99.5),
)

In [ ]:
cl_ref = hp.anafast(m_ref, lmax=lmax_out)
cl_ud = hp.anafast(m_ud, lmax=lmax_out)
cl_harm_default = hp.anafast(m_harm_default, lmax=lmax_out)
cl_harm_lmax_out = hp.anafast(m_harm_lmax_out, lmax=lmax_out)
cl_smooth_ud = hp.anafast(m_smooth_ud, lmax=lmax_out)
ell_out = np.arange(lmax_out + 1)

plt.figure(figsize=(8,4.5))
plt.loglog(ell_out[2:], cl_ref[2:], label='reference', lw=2)
plt.loglog(ell_out[2:], cl_ud[2:], label='ud_grade')
plt.loglog(ell_out[2:], cl_harm_default[2:], label='harmonic default')
plt.loglog(ell_out[2:], cl_harm_lmax_out[2:], label='harmonic lmax=3N-1')
plt.loglog(ell_out[2:], cl_smooth_ud[2:], label='smoothing + ud')
plt.xlabel('ell')
plt.ylabel('C_ell')
plt.title('Synthetic spectra vs reference')
plt.grid(alpha=0.25)
plt.legend()

In [ ]:
synthetic_metrics = {
    'ud_grade': compare_to_ref(m_ud, m_ref, cl_ud, cl_ref),
    'harmonic_default': compare_to_ref(m_harm_default, m_ref, cl_harm_default, cl_ref),
    'harmonic_lmax_3N-1': compare_to_ref(m_harm_lmax_out, m_ref, cl_harm_lmax_out, cl_ref),
    'smoothing_plus_ud': compare_to_ref(m_smooth_ud, m_ref, cl_smooth_ud, cl_ref),
}
synthetic_metrics

## Synthetic quality metrics (Matplotlib)
**Scope:** these scalar metrics are computed **only for the synthetic power-law experiment** (Section 1), where a known target-NSIDE truth map exists.


This section summarizes how each downgrade method matches the direct target-NSIDE reference map. All metrics are computed against that reference and shown as a 2x2 matrix of bars.

**Definition of `ref` used below**

- `ref` means the synthetic target-NSIDE map built directly from the same realization, i.e. `m_ref` from `alm_ref`.
- In pixel-space metrics, `ref` is `m_ref` evaluated on the same valid pixels as the test map.
- In spectrum-space metrics, `ref` is `cl_ref = hp.anafast(m_ref, lmax=lmax_out)`.
- In `std(ref)`, the denominator is `np.std(m_ref[good])`, where `good` is the finite-pixel mask used for the method map and `m_ref`.

### What each statistic means

- `RMSE / std(ref)`: `sqrt(mean((m - m_ref)^2)) / std(m_ref)` on valid pixels. This emphasizes larger deviations because errors are squared before averaging. Lower is better.
- `MAE / std(ref)`: `mean(abs(m - m_ref)) / std(m_ref)` on valid pixels. This is more robust to outliers than RMSE. Lower is better.
- `relative spectrum L2`: `||cl - cl_ref||_2 / ||cl_ref||_2` over multipoles used in the notebook (from `ell_min=2` to `lmax_out`). This measures spectral-shape mismatch in harmonic space. Lower is better.
- `1 - map correlation`: `1 - corr(m, m_ref)` on valid pixels. A perfect correlation gives zero; larger values indicate morphological disagreement. Lower is better.

These four statistics are complementary: RMSE/MAE quantify pixel-domain amplitude error, `relative spectrum L2` quantifies harmonic-domain mismatch, and `1 - correlation` focuses on pattern agreement independent of overall scaling.


In [ ]:
method_order = ["harmonic_lmax_3N-1", "harmonic_default", "smoothing_plus_ud", "ud_grade"]
method_labels = {
    "harmonic_lmax_3N-1": "harmonic (lmax=3N-1)",
    "harmonic_default": "harmonic (default lmax)",
    "smoothing_plus_ud": "smoothing + ud_grade",
    "ud_grade": "ud_grade",
}

metric_labels = {
    "rmse_rel_std": "RMSE / std(ref)",
    "mae_rel_std": "MAE / std(ref)",
    "spec_rel_l2": "relative spectrum L2",
    "one_minus_corr": "1 - map correlation",
}

metrics_plot = {}
for method, vals in synthetic_metrics.items():
    metrics_plot[method] = {
        "rmse_rel_std": vals["rmse_rel_std"],
        "mae_rel_std": vals["mae_rel_std"],
        "spec_rel_l2": vals["spec_rel_l2"],
        "one_minus_corr": 1.0 - vals["map_corr"],
    }

metric_order = ["rmse_rel_std", "mae_rel_std", "spec_rel_l2", "one_minus_corr"]

print("Synthetic metrics vs direct reference (lower is better):")
for method in method_order:
    row = ", ".join(
        f"{metric_labels[m]}={metrics_plot[method][m]:.3e}" for m in metric_order
    )
    print(f"  {method_labels[method]}: {row}")


### Absolute metrics matrix

The first matrix shows absolute values of each metric for each method on a log y-scale, always relative to `ref` (`m_ref` and `cl_ref` as defined above).

- Interpretation: compare bars within each panel; shorter bars indicate better agreement with `ref`.
- Why log scale: values can differ by orders of magnitude (especially for near-exact harmonic cases).
- Use this matrix to assess raw performance against the synthetic target-NSIDE truth, independent of any chosen baseline method.


In [ ]:
colors = ["#1b9e77", "#d95f02", "#7570b3", "#666666"]
x = np.arange(len(method_order))

fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
for ax, metric in zip(axes.flat, metric_order):
    values = np.array([metrics_plot[m][metric] for m in method_order], dtype=float)
    bars = ax.bar(x, values, color=colors)
    ax.set_yscale("log")
    ax.set_title(metric_labels[metric])
    ax.set_xticks(x, [method_labels[m] for m in method_order], rotation=20, ha="right")
    ax.set_ylabel("value (log scale)")
    ax.grid(axis="y", alpha=0.3)
    for b, v in zip(bars, values):
        ax.text(b.get_x() + b.get_width() / 2.0, v, f"{v:.2e}", ha="center", va="bottom", fontsize=8)

fig.suptitle("Absolute quality metrics matrix (all lower = better)")
plt.show()


### Relative metrics matrix (normalized to `ud_grade`)

The second matrix divides each absolute metric by the corresponding `ud_grade` value. Both numerator and denominator are still metrics computed versus `ref`; only the normalization changes.

- Ratio definition: `metric(method vs ref) / metric(ud_grade vs ref)`.
- Baseline line: dashed horizontal line at `1.0` marks parity with `ud_grade`.
- Values `< 1`: method improves on `ud_grade` for that metric.
- Values `> 1`: method is worse than `ud_grade` for that metric.
- Use this matrix to measure practical gain/loss versus the standard baseline, not just absolute error size.


In [ ]:
baseline = "ud_grade"
fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
for ax, metric in zip(axes.flat, metric_order):
    values = np.array([metrics_plot[m][metric] for m in method_order], dtype=float)
    ratio = values / metrics_plot[baseline][metric]
    bars = ax.bar(x, ratio, color=colors)
    ax.axhline(1.0, color="black", linestyle="--", linewidth=1)
    ax.set_yscale("log")
    ax.set_title(f"{metric_labels[metric]} / {method_labels[baseline]}")
    ax.set_xticks(x, [method_labels[m] for m in method_order], rotation=20, ha="right")
    ax.set_ylabel("ratio (log scale)")
    ax.grid(axis="y", alpha=0.3)
    for b, v in zip(bars, ratio):
        ax.text(b.get_x() + b.get_width() / 2.0, v, f"{v:.2g}x", ha="center", va="bottom", fontsize=8)

fig.suptitle("Relative quality metrics matrix vs ud_grade (values < 1 are better)")
plt.show()


In [ ]:
proj_panels(
    [m_ud - m_ref, m_harm_default - m_ref, m_harm_lmax_out - m_ref, m_smooth_ud - m_ref],
    ['ud - ref', 'harm(default) - ref', 'harm(lmax=3N-1) - ref', 'smooth+ud - ref'],
    cmap='RdBu_r',
    q=(0.1, 99.9),
    symmetric=True,
)


### Synthetic Mollweide differences vs reference
Same residuals shown as Mollweide maps for direct visual comparison.


In [ ]:
moll_diff_panels(
    [m_ud - m_ref, m_harm_default - m_ref, m_harm_lmax_out - m_ref, m_smooth_ud - m_ref],
    ['ud - ref', 'harm(default) - ref', 'harm(lmax=3N-1) - ref', 'smooth+ud - ref'],
    unit='arb.',
    q=(0.1, 99.9),
)


## 2) Real d10 353 GHz intensity template (direct download, no PySM)

In [ ]:
base_url = 'https://portal.nersc.gov/project/cmb/pysm-data'
relpath = 'dust_gnilc/gnilc_dust_template_nside2048_2023.02.10.fits'
url = f'{base_url}/{relpath}'

cache_dir = Path('/tmp/pysm_data')
cache_dir.mkdir(parents=True, exist_ok=True)
local_path = cache_dir / Path(relpath).name
if not local_path.exists():
    print('Downloading', url)
    urllib.request.urlretrieve(url, local_path)
print('Local file:', local_path)

In [ ]:
m_d10 = hp.read_map(local_path, field=0, dtype=np.float64)
nside_d10 = hp.get_nside(m_d10)
print('d10 nside:', nside_d10, 'npix:', m_d10.size)

In [ ]:
nside_out_real = 128
lmax_real = 3*nside_out_real - 1

d10_ud = hp.ud_grade(m_d10, nside_out=nside_out_real)
d10_harm_default = hp.harmonic_ud_grade(m_d10, nside_out=nside_out_real)
d10_harm_lmax_out = hp.harmonic_ud_grade(m_d10, nside_out=nside_out_real, lmax=lmax_real)
d10_smooth_ud = hp.ud_grade(hp.smoothing(m_d10, fwhm=np.radians(30/60)), nside_out=nside_out_real)

In [ ]:
proj_panels(
    [d10_ud, d10_harm_default, d10_harm_lmax_out, d10_smooth_ud],
    ['d10: ud_grade', 'd10: harmonic default', 'd10: harmonic lmax=3N-1', 'd10: smoothing + ud'],
    cmap='viridis',
    q=(1, 99),
)

### d10 residual maps (visual check)
For d10, there is no independent target-NSIDE ground-truth map analogous to `m_ref` in the synthetic case.
Therefore this section uses relative diagnostics (method-to-method residual maps and spectra) rather than the scalar truth-referenced metrics used in Section 1.


In [ ]:
d10_ref = d10_harm_lmax_out
proj_panels(
    [d10_ud - d10_ref, d10_harm_default - d10_ref, d10_smooth_ud - d10_ref],
    ['d10: ud - harm(lmax=3N-1)', 'd10: harm(default) - harm(lmax=3N-1)', 'd10: smooth+ud - harm(lmax=3N-1)'],
    cmap='RdBu_r',
    q=(0.2, 99.8),
    symmetric=True,
)


### d10 Mollweide differences vs harmonic reference
Mollweide residuals use the same symmetric color range across methods.


In [ ]:
moll_diff_panels(
    [d10_ud - d10_ref, d10_harm_default - d10_ref, d10_smooth_ud - d10_ref],
    ['d10: ud - harm(lmax=3N-1)', 'd10: harm(default) - harm(lmax=3N-1)', 'd10: smooth+ud - harm(lmax=3N-1)'],
    unit='MJy/sr',
    q=(0.2, 99.8),
)


In [ ]:
cl_d10_orig = hp.anafast(m_d10, lmax=lmax_real)
cl_d10_ud = hp.anafast(d10_ud, lmax=lmax_real)
cl_d10_harm_default = hp.anafast(d10_harm_default, lmax=lmax_real)
cl_d10_harm_lmax_out = hp.anafast(d10_harm_lmax_out, lmax=lmax_real)
cl_d10_smooth_ud = hp.anafast(d10_smooth_ud, lmax=lmax_real)
ell_real = np.arange(lmax_real + 1)

line_styles = {
    "original": "-",
    "ud_grade": "--",
    "harmonic_default": "-.",
    "harmonic_lmax": ":",
    "smoothing_ud": (0, (5, 2)),
}

plt.figure(figsize=(8,4.5))
plt.loglog(ell_real[2:], cl_d10_orig[2:], label="original d10 input", linestyle=line_styles["original"], linewidth=2.2)
plt.loglog(ell_real[2:], cl_d10_ud[2:], label="ud_grade", linestyle=line_styles["ud_grade"], linewidth=2.0)
plt.loglog(ell_real[2:], cl_d10_harm_default[2:], label="harmonic default", linestyle=line_styles["harmonic_default"], linewidth=2.0)
plt.loglog(ell_real[2:], cl_d10_harm_lmax_out[2:], label="harmonic lmax=3N-1", linestyle=line_styles["harmonic_lmax"], linewidth=2.4)
plt.loglog(ell_real[2:], cl_d10_smooth_ud[2:], label="smoothing + ud", linestyle=line_styles["smoothing_ud"], linewidth=2.0)
plt.xlabel("ell")
plt.ylabel("C_ell")
plt.title("d10 intensity spectra: original and downgraded maps")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()


For d10, the apparent spectral "drop" with harmonic downgrade is expected when `lmax` is conservative.
Using `lmax=3*nside_out-1` shifts the harmonic result toward the full target-bandlimit while still avoiding `ud_grade` alias leakage.

### d10 spectra with Planck Galactic mask (large cut)
We apply a Planck Galactic mask (`GAL060`, from `HFI_Mask_GalPlane-apo0_2048_R2.00`) to emphasize high-latitude behavior and reduce Galactic-plane dominance in the spectra.


In [ ]:
from astropy.io import fits

mask_filename = "HFI_Mask_GalPlane-apo0_2048_R2.00.fits"
mask_url = f"https://irsa.ipac.caltech.edu/data/Planck/release_2/ancillary-data/masks/{mask_filename}"
mask_path = Path(mask_filename)
if not mask_path.exists():
    urllib.request.urlretrieve(mask_url, mask_path)

# Large Galactic cut: GAL060 keeps ~60% of sky (apodized).
mask_field = "GAL060"
with fits.open(mask_path) as hdul:
    col_names = list(hdul[1].columns.names)
mask_field_idx = col_names.index(mask_field)

mask_hi = hp.read_map(mask_path, field=mask_field_idx, dtype=np.float64)

def mask_at_nside(mask, nside_out):
    nside_in = hp.get_nside(mask)
    if nside_in == nside_out:
        out = mask
    else:
        out = hp.ud_grade(mask, nside_out=nside_out)
    return np.clip(out, 0.0, 1.0)

def masked_cl(m, mask_base, lmax):
    mask = mask_at_nside(mask_base, hp.get_nside(m))
    return hp.anafast(m * mask, lmax=lmax) / np.mean(mask**2), float(np.mean(mask**2))

cl_d10_orig_mask, fsky_orig = masked_cl(m_d10, mask_hi, lmax_real)
cl_d10_ud_mask, fsky_lo = masked_cl(d10_ud, mask_hi, lmax_real)
cl_d10_harm_default_mask, _ = masked_cl(d10_harm_default, mask_hi, lmax_real)
cl_d10_harm_lmax_out_mask, _ = masked_cl(d10_harm_lmax_out, mask_hi, lmax_real)
cl_d10_smooth_ud_mask, _ = masked_cl(d10_smooth_ud, mask_hi, lmax_real)

print(
    f"Mask field: {mask_field} (column {mask_field_idx}); "
    f"f_sky_eff(nside={nside_d10})={fsky_orig:.3f}, "
    f"f_sky_eff(nside={nside_out_real})={fsky_lo:.3f}"
)

line_styles = {
    "original": "-",
    "ud_grade": "--",
    "harmonic_default": "-.",
    "harmonic_lmax": ":",
    "smoothing_ud": (0, (5, 2)),
}

plt.figure(figsize=(8,4.5))
plt.loglog(ell_real[2:], cl_d10_orig_mask[2:], label="original d10 input (masked)", linestyle=line_styles["original"], linewidth=2.2)
plt.loglog(ell_real[2:], cl_d10_ud_mask[2:], label="ud_grade (masked)", linestyle=line_styles["ud_grade"], linewidth=2.0)
plt.loglog(ell_real[2:], cl_d10_harm_default_mask[2:], label="harmonic default (masked)", linestyle=line_styles["harmonic_default"], linewidth=2.0)
plt.loglog(ell_real[2:], cl_d10_harm_lmax_out_mask[2:], label="harmonic lmax=3N-1 (masked)", linestyle=line_styles["harmonic_lmax"], linewidth=2.4)
plt.loglog(ell_real[2:], cl_d10_smooth_ud_mask[2:], label="smoothing + ud (masked)", linestyle=line_styles["smoothing_ud"], linewidth=2.0)
plt.xlabel("ell")
plt.ylabel("pseudo-C_ell / <w^2>")
plt.title(f"d10 masked spectra with Planck {mask_field} Galactic mask")
plt.grid(alpha=0.25)
plt.legend(fontsize=9)
plt.tight_layout()
